Exercises:
E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?


Trigram langauge model

In [6]:
words = open('names.txt','r').read().splitlines()

In [7]:
len(words)

32033

In [8]:
words[:4]

['emma', 'olivia', 'ava', 'isabella']

In [9]:
import torch

In [10]:
N = torch.zeros((27,27,27), dtype = torch.int32)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [11]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs,chs[1:],chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    N[ix1,ix2,ix3] += 1


p = (N+1).float()
p = p/p.sum(2,keepdims=True)


In [12]:
p[1,1].sum()

tensor(1.0000)

In [13]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  out = ['.','.']
  ix = 0
  while True:
    p1 = p[stoi[out[-2]],stoi[out[-1]]]
    ix = torch.multinomial(p1,num_samples=1,replacement=True,generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out[2:]))


ce.
za.
zogh.
uriana.
kaydnevonimittain.


In [14]:
log_likelihood = 0.0
n = 0

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs,chs[1:],chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    prob =p[ix1,ix2,ix3]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

print(f'{log_likelihood=}')
null = -log_likelihood
print(f'{null=}')
print(f'{null/n}')



log_likelihood=tensor(-410414.9688)
null=tensor(410414.9688)
2.092747449874878


In [15]:
xs ,ys = [],[]

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs,chs[1:],chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    xs.append((ix1,ix2))
    ys.append(ix3)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
#num = xs.nelement()
#print(f'{num}')




In [16]:
xs

tensor([[ 0,  5],
        [ 5, 13],
        [13, 13],
        ...,
        [26, 25],
        [25, 26],
        [26, 24]])

In [17]:
ys

tensor([13, 13,  1,  ..., 26, 24,  0])

In [18]:
g = torch.Generator().manual_seed(2147483647)
w = torch.randn((54,27) , generator=g, requires_grad=True)

In [19]:
import torch.nn.functional as F
for k in range(1000):
  xenc = F.one_hot(xs,num_classes=27).float()
  logits = xenc.view(-1,27*2) @ w
  counts = logits.exp()
  prob = counts / counts.sum(1,keepdim=True)
  loss = -prob[torch.arange(ys.shape[0]),ys].log().mean()

  w.grad = None
  loss.backward()

  w.data += -50 * w.grad


print(loss.item())


2.2381389141082764


In [20]:
from os.path import isjunction
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
  out = []
  ix = 0
  jx = 0
  while True:
    xenc = F.one_hot(torch.tensor([ix,jx]),num_classes=27).float()
    logits = xenc.view(-1,27*2) @ w
    counts = logits.exp()
    probs = counts/counts.sum(1,keepdims=True)
    ix1 = torch.multinomial(probs,num_samples=1,replacement=True,generator=g).item()
    out.append(itos[ix1])
    ix = jx
    jx =ix1
    if ix1 == 0:
      break
  print(''.join(out))


aexze.
ahallurailazitynnellin.
alia.
nallayn.
ka.
ar.
ra.
zyaubrtthrigotai.
iolielliaugie.
amda.


Training split/dev split/testsplit

E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [21]:
def build_dataset(words):
  xs ,ys = [],[]

  for w in words:
      chs = ['.'] + list(w) + ['.']
      for ch1,ch2,ch3 in zip(chs,chs[1:],chs[2:]):
         ix1 = stoi[ch1]
         ix2 = stoi[ch2]
         ix3 = stoi[ch3]
         xs.append([ix1,ix2])
         ys.append(ix3)
  xs=torch.tensor(xs)
  ys=torch.tensor(ys)
  return xs,ys

In [22]:
import random
random.seed(42)
random.shuffle(words)

In [23]:
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

xtr,ytr = build_dataset(words[:n1])
xdev,ydev = build_dataset(words[n1:n2])
xte,yte = build_dataset(words[n2:])


g = torch.Generator().manual_seed(2147483647)
w = torch.randn((27*2,27),generator=g,requires_grad=True)

In [24]:
train_loss = []
dev_loss = []


for i in range(100):

  xenc =  F.one_hot(xtr,num_classes=27).float()
  logits = xenc.view(-1,54) @ w
  counts = logits.exp()
  probs = counts/counts.sum(1,keepdims=True)
  loss_tr = -probs[torch.arange(ytr.shape[0]),ytr].log().mean() + 0.01 * (w**2).mean()

  if i>=90:
    with torch.no_grad():
      xenc = F.one_hot(xdev,num_classes=27).float()
      logits =xenc.view(-1,27*2) @ w
      counts = logits.exp()
      probs_dev = counts/counts.sum(1,keepdims=True)
      loss_dev= -probs_dev[torch.arange(ydev.shape[0]),ydev].log().mean()

      print(f'Training loss: {loss_tr} and devloss : {loss_dev}')
    train_loss.append(loss_tr.item())
    dev_loss.append(loss_dev.item())
  w.grad = None
  loss_tr.backward()
  w.data += -50 * w.grad


print("Mean of the last 10 training loss: ", sum(train_loss)/10)
print("Mean of the last 10 training loss: ", sum(dev_loss)/10)






Training loss: 2.2758302688598633 and devloss : 2.264760732650757
Training loss: 2.2754883766174316 and devloss : 2.264401435852051
Training loss: 2.275153875350952 and devloss : 2.264050245285034
Training loss: 2.274827003479004 and devloss : 2.263706684112549
Training loss: 2.2745068073272705 and devloss : 2.2633707523345947
Training loss: 2.2741942405700684 and devloss : 2.2630422115325928
Training loss: 2.273888349533081 and devloss : 2.2627205848693848
Training loss: 2.2735886573791504 and devloss : 2.2624056339263916
Training loss: 2.2732958793640137 and devloss : 2.2620980739593506
Training loss: 2.2730088233947754 and devloss : 2.261796236038208
Mean of the last 10 training loss:  2.274378228187561
Mean of the last 10 training loss:  2.263235259056091


In [25]:
with torch.no_grad():
      xenc = F.one_hot(xte,num_classes=27).float()
      logits =xenc.view(-1,27*2) @ w
      counts = logits.exp()
      probs_dev = counts/counts.sum(1,keepdims=True)
      loss = -probs_dev[torch.arange(yte.shape[0]),yte].log().mean()

      print(f'Test loss: ',loss.item())

Test loss:  2.267354965209961


E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

In [28]:
import torch
import torch.nn.functional as F

smoothnesses = [0.005, 0.0001, 0.0002, 0.0005]

best_dev_losses = {}   # tracks final dev loss for each smoothing value

for i in smoothnesses:
    g = torch.Generator().manual_seed(2147483647)
    w = torch.randn((27*2, 27), generator=g, requires_grad=True)
    print(f"--------------smoothing value of {i}--------------------")

    train_loss = []
    dev_loss = []

    for j in range(1000):
        # forward pass (train) — indexing instead of one-hot (E04)
        logits = w[:27][xtr[:,0]] + w[27:][xtr[:,1]]
        loss_tr = F.cross_entropy(logits, ytr) + i * (w**2).mean()   # E05: cross_entropy

        # dev loss (no regularization here)
        with torch.no_grad():
            logits_dev = w[:27][xdev[:,0]] + w[27:][xdev[:,1]]
            loss_dev = F.cross_entropy(logits_dev, ydev)

        train_loss.append(loss_tr.item())
        dev_loss.append(loss_dev.item())

        # backward + update
        w.grad = None
        loss_tr.backward()
        w.data += -50 * w.grad

    print("Final training loss: ", train_loss[-1])
    print("Final dev loss: ", dev_loss[-1])
    best_dev_losses[i] = dev_loss[-1]
    print("-------------------------------------------------------")

# pick the best smoothing value based on final dev loss
best_smoothing = min(best_dev_losses, key=best_dev_losses.get)
print("\n=========================================================")
print("Best smoothing value:", best_smoothing, "-> dev loss:", best_dev_losses[best_smoothing])
print("=========================================================\n")



--------------smoothing value of 0.005--------------------
Final training loss:  2.246255397796631
Final dev loss:  2.2384743690490723
-------------------------------------------------------
--------------smoothing value of 0.0001--------------------
Final training loss:  2.238456964492798
Final dev loss:  2.2377750873565674
-------------------------------------------------------
--------------smoothing value of 0.0002--------------------
Final training loss:  2.238637685775757
Final dev loss:  2.237783670425415
-------------------------------------------------------
--------------smoothing value of 0.0005--------------------
Final training loss:  2.2391746044158936
Final dev loss:  2.237811326980591
-------------------------------------------------------

Best smoothing value: 0.0001 -> dev loss: 2.2377750873565674



In [29]:
# retrain fresh with the best smoothing value
g = torch.Generator().manual_seed(2147483647)
w = torch.randn((27*2, 27), generator=g, requires_grad=True)

for j in range(1000):
    logits = w[:27][xtr[:,0]] + w[27:][xtr[:,1]]
    loss_tr = F.cross_entropy(logits, ytr) + best_smoothing * (w**2).mean()
    w.grad = None
    loss_tr.backward()
    w.data += -50 * w.grad

# evaluate on test set ONCE, at the very end
with torch.no_grad():
    logits_test = w[:27][xte[:,0]] + w[27:][xte[:,1]]
    loss_test = F.cross_entropy(logits_test, yte)

print("Final Test loss:", loss_test.item())

Final Test loss: 2.2434422969818115


E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

Replaced F.one_hot(x) @ w with direct indexing: w[:27][xtr[:,0]] + w[27:][xtr[:,1]] same result, since a one-hot vector times W just selects a row of W anyway, so indexing skips the wasted computation

In [32]:
smoothnesses = [0.0001, 0.0002,0.0005]

for i in smoothnesses:
  g = torch.Generator().manual_seed(2147483647)
  w = torch.randn((27*2,27),generator=g,requires_grad=True)
  print(f"--------------smooting value of {i}--------------------")
  train_loss = []
  dev_loss = []
  for j in range(1000):
   # xenc =  F.one_hot(xtr,num_classes=27).float()
    logits = w[:27][xtr[:,0]]+w[27:][xtr[:,1]]
    counts = logits.exp()
    probs = counts/counts.sum(1,keepdims=True)
    loss_tr = -probs[torch.arange(ytr.shape[0]),ytr].log().mean() + i * (w**2).mean()
    with torch.no_grad():
      #xenc = F.one_hot(xdev,num_classes=27).float()
      logits = w[:27][xdev[:,0]]+w[27:][xdev[:,1]]
      counts = logits.exp()
      probs_dev = counts/counts.sum(1,keepdims=True)
      loss_dev= -probs_dev[torch.arange(ydev.shape[0]),ydev].log().mean()
    train_loss.append(loss_tr.item())
    dev_loss.append(loss_dev.item())
    w.grad = None
    loss_tr.backward()
    w.data += -50 * w.grad
  print("Final train loss ",train_loss[-1])
  print("Final dev loss ",dev_loss[-1])


--------------smooting value of 0.0001--------------------
Final train loss  2.2384567260742188
Final dev loss  2.2377748489379883
--------------smooting value of 0.0002--------------------
Final train loss  2.238637924194336
Final dev loss  2.237783670425415
--------------smooting value of 0.0005--------------------
Final train loss  2.2391748428344727
Final dev loss  2.2378108501434326


E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

F.cross_entropy does the same softmax + log + averaging math you wrote by hand, just in one line it's a built-in shortcut, not a different calculation.
F.cross_entropy faster,less code to get wrong

In [54]:
g = torch.Generator().manual_seed(2147483647)
w = torch.randn((27*2, 27), generator=g, requires_grad=True)

train_loss = []
dev_loss = []

for i in range(3000):
    logits = w[:27][xtr[:,0]] + w[27:][xtr[:,1]]
    loss_tr = F.cross_entropy(logits, ytr) + 0.0001 * (w**2).mean()

    with torch.no_grad():
        logits_dev = w[:27][xdev[:,0]] + w[27:][xdev[:,1]]
        loss_dev = F.cross_entropy(logits_dev, ydev)

    train_loss.append(loss_tr.item())
    dev_loss.append(loss_dev.item())

    w.grad = None
    loss_tr.backward()
    w.data += -50 * w.grad

print("Final training loss: ", train_loss[-1])
print("Final dev loss: ", dev_loss[-1])

with torch.no_grad():
    logits_test = w[:27][xte[:,0]] + w[27:][xte[:,1]]
    loss_test = F.cross_entropy(logits_test, yte)

print("Test loss:", loss_test.item())

Final training loss:  2.2372195720672607
Final dev loss:  2.2375574111938477
Test loss: 2.242598533630371
